# Notebook 05 — Consolidation des Résultats Baseline (Phase 1)

**Milestone M-06 | Équipe complète**

Ce notebook :
1. Rassemble les métriques des 3 modèles (TF-IDF, BM25+, Sentence Transformers)
2. Produit le tableau comparatif
3. Analyse qualitative sur des requêtes représentatives
4. Synthèse des forces et faiblesses de chaque approche

## 1. Imports

In [ ]:
import json
import os
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Plus
from sentence_transformers import SentenceTransformer

sns.set_theme(style='whitegrid')

DATA_DIR    = '../data'
MODELS_DIR  = '../models'
OUTPUTS_DIR = '../outputs'

print('Imports OK')

## 2. Chargement des Résultats Pré-calculés

In [ ]:
with open(os.path.join(OUTPUTS_DIR, 'tfidf_results.pkl'), 'rb') as f:
    tfidf_results = pickle.load(f)

with open(os.path.join(OUTPUTS_DIR, 'bm25_results.pkl'), 'rb') as f:
    bm25_results = pickle.load(f)

with open(os.path.join(OUTPUTS_DIR, 'embeddings_results.pkl'), 'rb') as f:
    emb_results = pickle.load(f)

print('Résultats chargés :')
print(f'  TF-IDF            : {tfidf_results}')
print(f'  BM25+             : {bm25_results}')
print(f'  SentenceTransform : {emb_results}')

## 3. Tableau Comparatif Phase 1

In [ ]:
df_results = pd.DataFrame([tfidf_results, bm25_results, emb_results]).set_index('model')
df_results.columns = ['Recall@10', 'Precision@10', 'MRR']

print('=== Tableau Comparatif Phase 1 ===')
display(df_results.style
        .format('{:.4f}')
        .highlight_max(axis=0, color='lightgreen')
        .highlight_min(axis=0, color='#ffcccc')
        .set_caption('Métriques IR@10 — Phase 1 Baseline'))

In [ ]:
# Identification du meilleur modèle par métrique
print('Meilleur modèle par métrique :')
for col in df_results.columns:
    best = df_results[col].idxmax()
    val  = df_results[col].max()
    print(f'  {col:15s}: {best} ({val:.4f})')

### 3.1 Visualisation — Graphique en Barres Groupées

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x      = np.arange(len(df_results.columns))
width  = 0.25
models = df_results.index.tolist()
colors = ['steelblue', 'coral', 'seagreen']

for i, (model, color) in enumerate(zip(models, colors)):
    values = df_results.loc[model].values
    bars   = ax.bar(x + i * width, values, width, label=model, color=color, alpha=0.85, edgecolor='white')
    ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=9)

ax.set_xticks(x + width)
ax.set_xticklabels(df_results.columns, fontsize=12)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Comparaison des Modèles IR — Phase 1 Baseline', fontsize=14, fontweight='bold')
ax.legend(title='Modèle', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'baseline_comparison.png'), dpi=150)
plt.show()
print('Graphique sauvegardé.')

## 4. Reconstruction des Modèles pour Analyse Qualitative

In [ ]:
# Chargement du corpus
df_docs = pd.read_pickle(os.path.join(DATA_DIR, 'df_docs_preprocessed.pkl'))

with open(os.path.join(DATA_DIR, 'queries_train.json'), 'r', encoding='utf-8') as f:
    queries_train = json.load(f)

with open(os.path.join(DATA_DIR, 'qgts_train.json'), 'r', encoding='utf-8') as f:
    qgts_train = json.load(f)

with open(os.path.join(DATA_DIR, 'queries_test.json'), 'r', encoding='utf-8') as f:
    queries_test = json.load(f)

id_to_idx = {doc_id: idx for idx, doc_id in enumerate(df_docs['id'])}
idx_to_id = {idx: doc_id for doc_id, idx in id_to_idx.items()}

corpus_clean = df_docs['content_clean'].tolist()
corpus_raw   = df_docs['content'].tolist()

def get_relevant_ids(qgts, query_id):
    if query_id not in qgts:
        return []
    return [e['doc_id'] for e in qgts[query_id]['relevant_doc_ids']]

def build_query_text(q):
    parts = [q.get('text', ''), q.get('title', '')]
    if q.get('tags'):
        parts.append(' '.join(q['tags']))
    return ' '.join(p for p in parts if p).strip()

# TF-IDF
with open(os.path.join(MODELS_DIR, 'tfidf_vectorizer.pkl'), 'rb') as f:
    vectorizer = pickle.load(f)
with open(os.path.join(MODELS_DIR, 'tfidf_matrix.pkl'), 'rb') as f:
    tfidf_matrix = pickle.load(f)

# BM25+
with open(os.path.join(MODELS_DIR, 'bm25_index.pkl'), 'rb') as f:
    bm25 = pickle.load(f)

# Sentence Transformers
model_st = SentenceTransformer('all-MiniLM-L6-v2')
corpus_embeddings = np.load(os.path.join(MODELS_DIR, 'corpus_embeddings.npy'))

print('Tous les modèles chargés.')

In [ ]:
def search_tfidf(query, k=10):
    vec    = vectorizer.transform([query])
    scores = cosine_similarity(vec, tfidf_matrix).flatten()
    top_k  = np.argsort(scores)[::-1][:k]
    return {'topk_indices': top_k.tolist(), 'topk_scores': scores[top_k].tolist()}

def search_bm25(query, k=10):
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)
    top_k  = np.argsort(scores)[::-1][:k]
    return {'topk_indices': top_k.tolist(), 'topk_scores': scores[top_k].tolist()}

def search_embeddings(query, k=10):
    q_emb  = model_st.encode([query], convert_to_numpy=True)
    scores = cosine_similarity(q_emb, corpus_embeddings).flatten()
    top_k  = np.argsort(scores)[::-1][:k]
    return {'topk_indices': top_k.tolist(), 'topk_scores': scores[top_k].tolist()}

def recall_at_k(retrieved, relevant_ids, k, id_to_idx):
    rel_set = set(id_to_idx[r] for r in relevant_ids if r in id_to_idx)
    ret_set = set(retrieved[:k])
    return len(ret_set & rel_set) / max(len(rel_set), 1)

def mrr(retrieved, relevant_ids, id_to_idx):
    rel_set = set(id_to_idx[r] for r in relevant_ids if r in id_to_idx)
    for rank, idx in enumerate(retrieved, start=1):
        if idx in rel_set:
            return 1.0 / rank
    return 0.0

print('Fonctions de recherche et métriques prêtes.')

## 5. Analyse Qualitative sur 3 Requêtes Représentatives

In [ ]:
# 3 requêtes : facile (termes précis), ambiguë, difficile (paraphrase)
sample_indices = [0, len(queries_train)//3, 2*len(queries_train)//3]
sample_entries = [queries_train[i] for i in sample_indices]
labels = ['Facile', 'Ambiguë', 'Difficile']

search_fns = {
    'TF-IDF':               search_tfidf,
    'BM25+':                search_bm25,
    'SentenceTransformers': search_embeddings
}

for label, q_entry in zip(labels, sample_entries):
    q_text   = build_query_text(q_entry)
    q_id     = q_entry['id']
    relevant = get_relevant_ids(qgts_train, q_id)

    print(f'{"="*70}')
    print(f'[{label}] "{q_text[:75]}"')
    print(f'Docs pertinents : {len(relevant)} | IDs : {relevant[:2]}...')
    print()

    for model_name, fn in search_fns.items():
        result    = fn(q_text, k=5)
        retrieved = result['topk_indices']
        mrr_s     = mrr(retrieved, relevant, id_to_idx)
        rec_s     = recall_at_k(retrieved, relevant, 5, id_to_idx)
        print(f'  [{model_name}] MRR={mrr_s:.3f} | Recall@5={rec_s:.3f}')
        for idx, score in zip(result['topk_indices'][:3], result['topk_scores'][:3]):
            doc_id = idx_to_id[idx]
            marker = '✓' if doc_id in relevant else ' '
            title  = df_docs.iloc[idx]['title'][:50] or df_docs.iloc[idx]['text'][:50]
            print(f"    {marker} [{score:.3f}] {doc_id} — {title}")
    print()

## 6. Synthèse — Forces et Faiblesses

| Modèle | Forces | Faiblesses |
|--------|--------|------------|
| **TF-IDF** | Rapide, simple, explicable. Efficace sur des requêtes exactes avec termes du corpus. | Sensible à la synonymie et polysémie. Pénalise les documents longs. Dépend du preprocessing. |
| **BM25+** | Normalisation par longueur plus fine. Non-zéro pour les termes présents (delta). Paramétrable. | Même problème lexical que TF-IDF. Pas de compréhension sémantique. |
| **SentenceTransformers** | Capture la sémantique et les synonymes. Robuste aux paraphrases. Multilingue possible. | Plus lent à l'encodage. Nécessite GPU pour corpus large. Moins explicable. |

In [ ]:
# Affichage final du tableau récapitulatif
print('\n=== TABLEAU RÉCAPITULATIF PHASE 1 ===')
print(df_results.to_string(float_format='{:.4f}'.format))
print()
print('Fichiers produits :')
for f in sorted(os.listdir(OUTPUTS_DIR)):
    print(f'  outputs/{f}')

## 7. Conclusion Phase 1

La Phase 1 établit une base de comparaison solide avec trois approches complémentaires :

1. **TF-IDF** — référence classique, rapide et interprétable
2. **BM25+** — amélioration de TF-IDF avec meilleure normalisation par longueur
3. **Sentence Transformers** — approche sémantique dense, plus robuste aux variations linguistiques

**Prochaine étape — Phase 2 :**
- Entraîner un classifieur de catégories (M-07)
- Intégrer ce classifieur dans un pipeline de re-ranking (M-08)
- Comparer les résultats Phase 2 vs Phase 1 (M-09)

In [ ]:
import csv

# Choisir le meilleur modèle selon MRR
best_model_name = df_results['MRR'].idxmax()
best_fn = {'TF-IDF': search_tfidf, 'BM25+': search_bm25,
           'SentenceTransformers': search_embeddings}[best_model_name]

print(f'Modèle sélectionné pour la soumission : {best_model_name}')

K_SUBMIT = 100  # Kaggle attend généralement top-100
rows = []

for q_entry in queries_test:
    q_id   = q_entry['id']
    q_text = build_query_text(q_entry)
    q_cat  = q_entry.get('category', '')

    result   = best_fn(q_text, k=K_SUBMIT)
    doc_ids  = [idx_to_id[idx] for idx in result['topk_indices']]

    rows.append({
        'query_id':         q_id,
        'relevant_doc_ids': str(doc_ids),
        'category':         q_cat
    })

submission_path = os.path.join(OUTPUTS_DIR, 'submission.csv')
pd.DataFrame(rows).to_csv(submission_path, index=False)
print(f'Soumission générée : {submission_path}  ({len(rows)} requêtes × top-{K_SUBMIT})')
pd.DataFrame(rows).head(3)

## 7. Génération de la Soumission Kaggle

Utilise le meilleur modèle Phase 1 pour générer `submission.csv` sur les requêtes de test.

Format attendu : `query_id, relevant_doc_ids, category`